In [ ]:
import sys, os, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.colors import TwoSlopeNorm

# ── Project root ─────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'toy_domain'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'MedGrid'))

import med_grid_env  # registers MedGrid-v0 and MedGrid-discrete-v0
import gymnasium as gym

# ── Run to analyse ───────────────────────────────────────────────────────────
RUN_NAME = 'medgrid_iqn_continuous'   # change to match -info flag used during training
RUN_DIR  = os.path.join(PROJECT_ROOT, 'runs', RUN_NAME)

## 1. Ground truth regions

MedGrid is a **10×10** continuous gridworld with:
* **Red (Death):** quarter-circles of radius 3 at bottom-left $(0,0)$ and top-right $(10,10)$.
* **Yellow (Trap):** annular bands $3 < d \le 5$ around each death zone corner.
* **Blue (Recovery):** a rotated ellipse centred at $(5,5)$, semi-major $a=3$, semi-minor $b=1.5$, tilted $-45°$.
* **White (Neutral):** everything else.
* **Start:** bottom-right corner $(10, 0)$.

In [ ]:
SIZE = 10.0
DEATH_R = 3.0
TRAP_R  = 5.0
ELLIPSE_A = 3.0
ELLIPSE_B = 1.5
RESOLUTION = 400   # pixels per axis for the rasterised background


def classify_point(x, y):
    """Return region label for a single (x, y) point."""
    d_bl = np.sqrt(x**2 + y**2)
    d_tr = np.sqrt((x - 10)**2 + (y - 10)**2)

    if d_bl <= DEATH_R or d_tr <= DEATH_R:
        return 'death'
    if d_bl <= TRAP_R or d_tr <= TRAP_R:
        return 'trap'
    dx, dy = x - 5.0, y - 5.0
    if ((dx - dy)**2 / (2 * ELLIPSE_A**2) + (dx + dy)**2 / (2 * ELLIPSE_B**2)) <= 1.0:
        return 'recovery'
    return 'neutral'


def build_region_image(resolution=RESOLUTION):
    """Rasterise the region map into an RGBA image for imshow."""
    COLORS = {
        'death':    np.array([231,  76,  60, 200], dtype=np.uint8),
        'trap':     np.array([241, 196,  15, 200], dtype=np.uint8),
        'recovery': np.array([ 52, 152, 219, 200], dtype=np.uint8),
        'neutral':  np.array([255, 255, 255, 255], dtype=np.uint8),
    }
    xs = np.linspace(0, SIZE, resolution)
    ys = np.linspace(0, SIZE, resolution)
    img = np.zeros((resolution, resolution, 4), dtype=np.uint8)
    for j, y in enumerate(ys):
        for i, x in enumerate(xs):
            img[j, i] = COLORS[classify_point(x, y)]
    return img


def plot_ground_truth(ax=None, resolution=RESOLUTION):
    """Draw the MedGrid region layout. Returns (fig, ax)."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    else:
        fig = ax.figure

    img = build_region_image(resolution)
    ax.imshow(img, origin='lower', extent=[0, SIZE, 0, SIZE], aspect='equal')

    for i in range(11):
        ax.axhline(i, color='#bdc3c7', lw=0.5, zorder=2)
        ax.axvline(i, color='#bdc3c7', lw=0.5, zorder=2)

    ax.plot(1.5, 1.5, 'rx', markersize=10, markeredgewidth=2,
            label='Trap1 target (1.5, 1.5)', zorder=4)
    ax.plot(8.5, 8.5, 'rx', markersize=10, markeredgewidth=2,
            label='Trap2 target (8.5, 8.5)', zorder=4)
    ax.plot(10, 0, 'g*', markersize=16, label='Start (10, 0)', zorder=5)

    legend_patches = [
        mpatches.Patch(color='#e74c3c', label='Death (red) — r = -1, terminal'),
        mpatches.Patch(color='#f1c40f', label='Trap (yellow) — r = 0, forced step'),
        mpatches.Patch(color='#3498db', label='Recovery (blue) — r = +1, terminal'),
        mpatches.Patch(color='white',   label='Neutral (white) — r = 0'),
        plt.Line2D([0],[0], marker='*', color='w', markerfacecolor='g',
                   markersize=12, label='Start (10, 0)'),
        plt.Line2D([0],[0], marker='x', color='r',
                   markersize=8, markeredgewidth=2, label='Trap targets'),
    ]
    ax.legend(handles=legend_patches, loc='center', fontsize=7,
              framealpha=0.85, bbox_to_anchor=(0.5, 0.5))

    ax.set_xlim(0, SIZE)
    ax.set_ylim(0, SIZE)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('MedGrid — Ground Truth Regions')
    return fig, ax


print('Rasterising region map...')
fig, ax = plot_ground_truth()
plt.tight_layout()
plt.show()

## 2. Load trained agents

Run training first:
```bash
cd toy_domain
../.venv/bin/python run.py -env MedGrid -action_mode continuous \
    -agent iqn -ded -frames 500000 -info medgrid_iqn_continuous
```

In [ ]:
agent, qd, qr = None, None, None
try:
    with open(os.path.join(RUN_DIR, f'{RUN_NAME}_agent.pkl'), 'rb') as f:
        agent = pickle.load(f)
    print('Loaded main agent  (ContinuousIQN_Agent with Gaussian actor)')
    with open(os.path.join(RUN_DIR, f'{RUN_NAME}_Qd.pkl'), 'rb') as f:
        qd = pickle.load(f)
    with open(os.path.join(RUN_DIR, f'{RUN_NAME}_Qr.pkl'), 'rb') as f:
        qr = pickle.load(f)
    print('Loaded Qd and Qr networks')
except FileNotFoundError as e:
    print(f'Agents not found ({e}) — train first with the command above.')

## 3. CVaR Q-value heatmaps over action space at a fixed state

For a fixed state we sweep a regular grid over the **10×10 action space** and compute the
**lower CVaR** of the IQN quantile distribution at each action.

`ContinuousIQN.forward(state, action, num_tau)` returns `(quantiles, taus)` with
`quantiles` shaped `(batch, num_tau, 1)` — the same contract as the discrete IQN,
so the heatmap code is identical.

In [ ]:
HMAP_RESOLUTION = 50
N_TAU = 32
ETA   = 0.25
DEVICE = 'cpu'
FIXED_STATE = [10.0, 0.0]   # start corner

N_KEEP = max(1, int(ETA * N_TAU))
print(f'CVaR: keeping {N_KEEP}/{N_TAU} quantiles (eta={ETA})')
print(f'Fixed state: {FIXED_STATE}')


def compute_action_heatmap(ded_agent, state=FIXED_STATE, resolution=HMAP_RESOLUTION,
                            n_tau=N_TAU, eta=ETA, device=DEVICE):
    """Sweep a grid of actions for a fixed state, return CVaR Q heatmap."""
    axs = np.linspace(0.1, 9.9, resolution, dtype=np.float32)
    ays = np.linspace(0.1, 9.9, resolution, dtype=np.float32)
    AX, AY = np.meshgrid(axs, ays)
    actions = np.stack([AX.ravel(), AY.ravel()], axis=1)  # (R², 2)
    n_actions = len(actions)

    s = np.array([state], dtype=np.float32)
    states = np.tile(s, (n_actions, 1))            # (R², 2)

    n_keep = max(1, int(eta * n_tau))
    network = ded_agent.qnetwork_local.to(device)
    network.eval()

    s_t = torch.from_numpy(states).to(device)
    a_t = torch.from_numpy(actions).to(device)

    with torch.no_grad():
        quantiles, _ = network.forward(s_t, a_t, num_tau=n_tau)
        # quantiles: (R², n_tau, 1)

    q = quantiles.squeeze(-1).cpu().numpy()   # (R², n_tau)
    q_sorted = np.sort(q, axis=1)             # ascending
    cvar = q_sorted[:, :n_keep].mean(axis=1)  # (R²,)
    return cvar.reshape(resolution, resolution), axs, ays


def plot_action_heatmap(heatmap, axs, ays, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    else:
        fig = ax.figure
    norm = TwoSlopeNorm(vmin=heatmap.min(), vcenter=0, vmax=max(heatmap.max(), 1e-6))
    im = ax.imshow(heatmap, origin='lower',
                   extent=[axs[0], axs[-1], ays[0], ays[-1]],
                   cmap='RdBu', norm=norm, aspect='equal')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='CVaR Q')
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_xlabel('action x'); ax.set_ylabel('action y')
    ax.set_title(title)
    return fig, ax

In [ ]:
if qd is not None and qr is not None:
    print('Computing Qd heatmap...')
    heatmap_d, axs_h, ays_h = compute_action_heatmap(qd)
    print(f'  range: [{heatmap_d.min():.3f}, {heatmap_d.max():.3f}]')

    print('Computing Qr heatmap...')
    heatmap_r, _, _ = compute_action_heatmap(qr)
    print(f'  range: [{heatmap_r.min():.3f}, {heatmap_r.max():.3f}]')

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    plot_ground_truth(ax=axes[0], resolution=200)
    plot_action_heatmap(heatmap_d, axs_h, ays_h,
                        title=f'$Q_d$ CVaR (η={ETA}) @ {FIXED_STATE}', ax=axes[1])
    plot_action_heatmap(heatmap_r, axs_h, ays_h,
                        title=f'$Q_r$ CVaR (η={ETA}) @ {FIXED_STATE}', ax=axes[2])
    fig.suptitle(f'MedGrid — {RUN_NAME}  |  action space at state {FIXED_STATE}', fontsize=13)
    plt.tight_layout()
    out_path = os.path.join(RUN_DIR, 'medgrid_continuous_ded_action_heatmaps.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to', out_path)
else:
    print('Skipped — no trained agents loaded.')

## 4. Actor policy — mean action vector field over state space

The continuous agent's Gaussian actor outputs a mean action $\mu(s)$ for every state.
Here we rasterise the **state space** and plot the mean action as an arrow at each grid point,
overlaid on the ground-truth region map.

This visualises what the policy has learned: arrows pointing toward the recovery zone are good;
arrows pointing into death/trap zones indicate failure modes.

In [ ]:
STATE_RESOLUTION = 20   # grid points per axis for state-space sweep


def compute_actor_field(main_agent, resolution=STATE_RESOLUTION, device=DEVICE):
    """Return mean actions of the Gaussian actor on a state-space grid.

    Returns
    -------
    sx, sy : 1-D arrays of state grid coordinates
    mu_x, mu_y : (resolution, resolution) mean action components
    """
    assert hasattr(main_agent, 'actor_local'), \
        'Agent has no actor_local — was it trained with use_actor=True (continuous mode)?'

    sx = np.linspace(0.25, 9.75, resolution, dtype=np.float32)
    sy = np.linspace(0.25, 9.75, resolution, dtype=np.float32)
    SX, SY = np.meshgrid(sx, sy)
    states = np.stack([SX.ravel(), SY.ravel()], axis=1)  # (R², 2)

    actor = main_agent.actor_local.to(device)
    actor.eval()
    s_t = torch.from_numpy(states).to(device)
    with torch.no_grad():
        mu, _ = actor.sample(s_t)   # sample() returns (action, log_prob)
    actor.train()

    mu_np = mu.cpu().numpy()  # (R², 2)
    mu_x = mu_np[:, 0].reshape(resolution, resolution)
    mu_y = mu_np[:, 1].reshape(resolution, resolution)
    return sx, sy, mu_x, mu_y


if agent is not None and hasattr(agent, 'actor_local'):
    print('Computing actor mean-action field...')
    sx, sy, mu_x, mu_y = compute_actor_field(agent)

    fig, ax = plt.subplots(figsize=(7, 7))
    plot_ground_truth(ax=ax, resolution=200)

    # normalise arrow length for display
    mag = np.sqrt(mu_x**2 + mu_y**2) + 1e-8
    scale = 0.45  # arrow length in data units
    SX, SY = np.meshgrid(sx, sy)
    ax.quiver(
        SX, SY,
        (mu_x / mag) * scale,
        (mu_y / mag) * scale,
        color='black', alpha=0.7,
        angles='xy', scale_units='xy', scale=1,
        width=0.003, headwidth=4, headlength=5,
        zorder=5,
    )
    ax.set_title(f'Actor Mean-Action Field — {RUN_NAME}', fontsize=12)
    plt.tight_layout()
    out_path = os.path.join(RUN_DIR, 'medgrid_continuous_actor_field.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to', out_path)
elif agent is not None:
    print('Skipped — loaded agent has no actor (not trained in continuous mode).')
else:
    print('Skipped — no trained agent loaded.')

## 5. State-space heatmaps — Q_d and Q_r at the actor's chosen action

For every state on a grid, query the actor for its mean action $\mu(s)$, then evaluate
$Q_d(s, \mu(s))$ and $Q_r(s, \mu(s))$ using CVaR.  
This shows **how dangerous / recoverable the policy's own actions are** across the state space.

In [ ]:
STATE_HMAP_RES = 40


def compute_state_heatmap(main_agent, ded_agent, resolution=STATE_HMAP_RES,
                          n_tau=N_TAU, eta=ETA, device=DEVICE):
    """For each state on a grid, use actor μ(s) as action, return CVaR Q heatmap."""
    sx = np.linspace(0.25, 9.75, resolution, dtype=np.float32)
    sy = np.linspace(0.25, 9.75, resolution, dtype=np.float32)
    SX, SY = np.meshgrid(sx, sy)
    states = np.stack([SX.ravel(), SY.ravel()], axis=1)  # (R², 2)

    # Get actor mean actions
    actor = main_agent.actor_local.to(device)
    actor.eval()
    s_t = torch.from_numpy(states).to(device)
    with torch.no_grad():
        actions, _ = actor.sample(s_t)   # (R², 2)
    actor.train()

    # Evaluate DeD network
    n_keep = max(1, int(eta * n_tau))
    network = ded_agent.qnetwork_local.to(device)
    network.eval()
    with torch.no_grad():
        quantiles, _ = network.forward(s_t, actions, num_tau=n_tau)
    network.train()

    q = quantiles.squeeze(-1).cpu().numpy()   # (R², n_tau)
    q_sorted = np.sort(q, axis=1)
    cvar = q_sorted[:, :n_keep].mean(axis=1)
    return cvar.reshape(resolution, resolution), sx, sy


def plot_state_heatmap(heatmap, sx, sy, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    else:
        fig = ax.figure
    norm = TwoSlopeNorm(vmin=heatmap.min(), vcenter=0, vmax=max(heatmap.max(), 1e-6))
    im = ax.imshow(heatmap, origin='lower',
                   extent=[sx[0], sx[-1], sy[0], sy[-1]],
                   cmap='RdBu', norm=norm, aspect='equal')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='CVaR Q')
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)
    ax.set_xlabel('state x'); ax.set_ylabel('state y')
    ax.set_title(title)
    return fig, ax


if agent is not None and qd is not None and qr is not None and hasattr(agent, 'actor_local'):
    print('Computing state-space Qd heatmap under actor policy...')
    sh_d, sx_h, sy_h = compute_state_heatmap(agent, qd)
    print(f'  Qd range: [{sh_d.min():.3f}, {sh_d.max():.3f}]')

    print('Computing state-space Qr heatmap under actor policy...')
    sh_r, _, _ = compute_state_heatmap(agent, qr)
    print(f'  Qr range: [{sh_r.min():.3f}, {sh_r.max():.3f}]')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_ground_truth(ax=axes[0], resolution=200)
    plot_state_heatmap(sh_d, sx_h, sy_h,
                       title=f'$Q_d(s, \\mu(s))$ CVaR (η={ETA})', ax=axes[1])
    plot_state_heatmap(sh_r, sx_h, sy_h,
                       title=f'$Q_r(s, \\mu(s))$ CVaR (η={ETA})', ax=axes[2])
    fig.suptitle(f'MedGrid — {RUN_NAME}  |  state-space DeD values under actor policy', fontsize=13)
    plt.tight_layout()
    out_path = os.path.join(RUN_DIR, 'medgrid_continuous_state_ded_heatmaps.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to', out_path)
else:
    print('Skipped — agents not loaded or agent has no actor.')

## 6. Combined summary

Side-by-side: ground truth | actor field | $Q_d$ state heatmap | $Q_r$ state heatmap.

In [ ]:
if (agent is not None and qd is not None and qr is not None
        and hasattr(agent, 'actor_local')):

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    # Ground truth
    plot_ground_truth(ax=axes[0], resolution=200)

    # Actor field
    sx2, sy2, mu_x2, mu_y2 = compute_actor_field(agent, resolution=15)
    plot_ground_truth(ax=axes[1], resolution=200)
    SX2, SY2 = np.meshgrid(sx2, sy2)
    mag2 = np.sqrt(mu_x2**2 + mu_y2**2) + 1e-8
    scale2 = 0.4
    axes[1].quiver(SX2, SY2,
                   (mu_x2/mag2)*scale2, (mu_y2/mag2)*scale2,
                   color='black', alpha=0.7,
                   angles='xy', scale_units='xy', scale=1,
                   width=0.003, headwidth=4, headlength=5, zorder=5)
    axes[1].set_title('Actor Mean-Action Field')

    # DeD state heatmaps
    plot_state_heatmap(sh_d, sx_h, sy_h,
                       title=f'$Q_d(s,\\mu(s))$ CVaR (η={ETA})', ax=axes[2])
    plot_state_heatmap(sh_r, sx_h, sy_h,
                       title=f'$Q_r(s,\\mu(s))$ CVaR (η={ETA})', ax=axes[3])

    fig.suptitle(f'MedGrid Continuous — {RUN_NAME}', fontsize=14)
    plt.tight_layout()
    out_path = os.path.join(RUN_DIR, 'medgrid_continuous_summary.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to', out_path)
else:
    print('Skipped — agents not loaded.')